In [1]:
import geopandas as gpd
import pandas as pd 

In [2]:
rc_lookups = pd.read_csv("output/rc_lookups.csv")
rc_ldc =pd.read_csv("output/retail_centers_with_ldc_retailN.csv")

In [4]:
rc_lookups

,RC_ID,geometry,LAD24NM,CountyName,RegionName,gsscode
0,6025,MULTIPOLYGON (((440445.7388613202 367018.09139...,North East Derbyshire,Derbyshire,East Midlands,E63008766
1,1193,MULTIPOLYGON (((439116.7765939621 363407.88751...,North East Derbyshire,Derbyshire,East Midlands,E63008809
2,6644,MULTIPOLYGON (((194309.88097327633 76200.71289...,Cornwall,Cornwall,South West,E63013978
3,6026,MULTIPOLYGON (((439008.4176877096 364010.85388...,North East Derbyshire,Derbyshire,East Midlands,E63008809
4,7407,MULTIPOLYGON (((438294.8993448653 371476.61951...,Chesterfield,Derbyshire,East Midlands,E63008717
...,...,...,...,...,...,...
9617,2844,MULTIPOLYGON (((111614.72348488995 512113.4056...,"Armagh City, Banbridge and Craigavon",Armagh,NI,NI
9618,4025,MULTIPOLYGON (((578163.6762067255 233984.52018...,Braintree,Essex,East of England,E63011054
9619,2676,MULTIPOLYGON (((401222.3074132583 90489.386812...,"Bournemouth, Christchurch and Poole",Dorset,South West,E63013860
9620,3757,MULTIPOLYGON (((255356.56308415128 666668.6817...,Glasgow City,City of glasgow,Scotland,S45001180


In [6]:
rc = rc_ldc.merge(
     rc_lookups.drop(columns=['geometry']),
    on='RC_ID',
    how='left'
)
rc

,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,LocalAuthorityNM_2,LocalAuthorityNM_3,geometry,LDC_retailN,LAD24NM,CountyName,RegionName,gsscode
0,6025,16,5,7.0,6.0,0.0,3.0,['Asian Restaurant' 'Electronics Store' 'Café'...,centre,North East Derbyshire,0,0,MULTIPOLYGON (((440445.7388613202 367018.09139...,2.0,North East Derbyshire,Derbyshire,East Midlands,E63008766
1,1193,131,71,86.0,33.0,4.0,8.0,"['Electronics Store' ""Doctor's Office"" 'Drugst...",centre,North East Derbyshire,0,0,MULTIPOLYGON (((439116.7765939621 363407.88751...,131.0,North East Derbyshire,Derbyshire,East Midlands,E63008809
2,6644,14,11,11.0,2.0,1.0,0.0,['Electronics Store' 'Hardware Store' 'Conveni...,centre,Cornwall,0,0,MULTIPOLYGON (((194309.88097327633 76200.71289...,10.0,Cornwall,Cornwall,South West,E63013978
3,6026,16,44,11.0,3.0,2.0,0.0,['Gym' 'Gastropub' 'Brewery' 'Funeral Home' 'B...,centre,North East Derbyshire,0,0,MULTIPOLYGON (((439008.4176877096 364010.85388...,17.0,North East Derbyshire,Derbyshire,East Midlands,E63008809
4,7407,12,4,6.0,3.0,1.0,2.0,"['Hardware Store' ""Doctor's Office"" 'Indian Re...",centre,Chesterfield,0,0,MULTIPOLYGON (((438294.8993448653 371476.61951...,8.0,Chesterfield,Derbyshire,East Midlands,E63008717
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9617,2844,46,16,32.0,13.0,1.0,0.0,['Convenience Store' 'Discount Store' 'Restaur...,retail_park,"Armagh City, Banbridge and Craigavon",0,0,MULTIPOLYGON (((111614.72348488995 512113.4056...,59.0,"Armagh City, Banbridge and Craigavon",Armagh,NI,NI
9618,4025,29,22,17.0,8.0,2.0,2.0,['Drugstore' 'Pet Service' 'Convenience Store'...,retail_park,Braintree,0,0,MULTIPOLYGON (((578163.6762067255 233984.52018...,14.0,Braintree,Essex,East of England,E63011054
9619,2676,50,11,34.0,14.0,2.0,0.0,['Textiles Store' 'Electronics Store' 'Sportin...,retail_park,"Bournemouth, Christchurch and Poole",0,0,MULTIPOLYGON (((401222.3074132583 90489.386812...,46.0,"Bournemouth, Christchurch and Poole",Dorset,South West,E63013860
9620,3757,32,22,26.0,6.0,0.0,0.0,['Sporting Goods Retail' 'Convenience Store' '...,retail_park,Glasgow City,0,0,MULTIPOLYGON (((255356.56308415128 666668.6817...,50.0,Glasgow City,City of glasgow,Scotland,S45001180


In [9]:
rc[rc['gsscode'].isna()]['centre_type'].value_counts()  

centre_type
centre         83
retail_park     8
Name: count, dtype: int64

In [11]:
# Assuming your dataframe is called 'rc_df'
rc_df = rc.copy()
# 1. Flag Largest in Region
rc_df['is_largest_region'] = rc_df.groupby('RegionName')['LDC_retailN'].transform('max') == rc_df['LDC_retailN']

# 2. Flag Largest in County
rc_df['is_largest_county'] = rc_df.groupby('CountyName')['LDC_retailN'].transform('max') == rc_df['LDC_retailN']

# 3. Flag Largest in Local Authority
rc_df['is_largest_lad'] = rc_df.groupby('LAD24NM')['LDC_retailN'].transform('max') == rc_df['LDC_retailN']

# 4. Flag Largest in Built-up Area (gsscode)
rc_df['is_largest_bua'] = rc_df.groupby('gsscode')['LDC_retailN'].transform('max') == rc_df['LDC_retailN']

In [13]:
rc_df['is_largest_bua'].value_counts()

is_largest_bua
False    7298
True     2324
Name: count, dtype: int64

In [14]:
def apply_nature_rules(row):
    # Tier 1: Regional Centre
    if row['is_largest_region']:
        return "Regional Centre"
    
    # Tier 2: Major Town Centre (Largest in County & >= 350)
    if row['is_largest_county']:
        if row['LDC_retailN'] >= 350:
            return "Major Town Centre"
        # Note: If Largest in County but < 350, it drops to Tier 3 logic
    
    # Tier 3: Town Centre
    # Largest in County (regardless of size) OR Rank 1 in LAD OR simply > 250 units
    if row['is_largest_county'] or row['is_largest_lad'] or row['LDC_retailN'] >= 250:
        return "Town Centre"
    
    # Tier 4 & 5: Market Town vs District Centre
    # Both require >= 150 units, but Market Town must be the "King" of its BUA
    if row['LDC_retailN'] >= 150:
        if row['is_largest_bua']:
            return "Market Town"
        else:
            return "District Centre"
    
    # Tier 6: Local Centres
    if 100 <= row['LDC_retailN'] < 150:
        return "Local Centre"
    
    # Tier 7: Small Local Centres
    return "Small Local Centre"

# Apply to your dataframe
rc_df['Classification_2025'] = rc_df.apply(apply_nature_rules, axis=1)

In [15]:
rc_df['Classification_2025'].value_counts()

Classification_2025
Small Local Centre    8245
Town Centre            512
Local Centre           399
Market Town            214
District Centre        186
Major Town Centre       54
Regional Centre         12
Name: count, dtype: int64